In [42]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd
from datetime import datetime

dateipfad = 'Expl_few_beijing.parquet'

# Schema und Spaltennamen
schema = pq.read_schema(dateipfad)
print("Alle Spaltennamen:", schema.names)

# Passen Sie die Spaltennamen ggf. an
col_datetime = 'datetime'   # z.B. 'timestamp'
col_sector   = 'sector'     # z.B. 'Sektor' oder 'sector_code'

# Typ der datetime-Spalte ermitteln
datetime_type = schema.field(col_datetime).type

# Dataset öffnen
dataset = ds.dataset(dateipfad, format='parquet')

# Filter 1: datetime == '2010-01-04 14:00:00'
dt = datetime.strptime('2010-01-04 14:00:00', '%Y-%m-%d %H:%M:%S')
target_dt = pa.scalar(dt, type=datetime_type)
filter_dt = pc.field(col_datetime) == target_dt

# Filter 2: sector != 'POWER_INDUSTRY'
filter_sector = pc.field(col_sector) != 'TRANSPORT'

# --- Kombinierte Bedingung mit & (UND) ---
filter_expr = filter_dt & filter_sector

# Gefilterte Daten laden
table = dataset.to_table(filter=filter_expr)
df = table.to_pandas()

print(f"Gefundene Zeilen: {len(df)}")
print(df.to_string())


print(df[col_sector].unique())

Alle Spaltennamen: ['city', 'datetime', 'pollutant', 'lat', 'lon', 'sector', 'activity_code', 'emissions_kg']
Gefundene Zeilen: 46662
          city            datetime pollutant    lat     lon          sector activity_code  emissions_kg
0      Beijing 2010-01-04 14:00:00     PM2.5  36.35  111.45  POWER_INDUSTRY       DEFAULT  0.000000e+00
1      Beijing 2010-01-04 14:00:00     PM2.5  36.35  111.55  POWER_INDUSTRY       DEFAULT  0.000000e+00
2      Beijing 2010-01-04 14:00:00     PM2.5  36.35  111.65  POWER_INDUSTRY       DEFAULT  0.000000e+00
3      Beijing 2010-01-04 14:00:00     PM2.5  36.35  111.75  POWER_INDUSTRY       DEFAULT  1.318573e-01
4      Beijing 2010-01-04 14:00:00     PM2.5  36.35  111.85  POWER_INDUSTRY       DEFAULT  0.000000e+00
5      Beijing 2010-01-04 14:00:00     PM2.5  36.35  111.95  POWER_INDUSTRY       DEFAULT  0.000000e+00
6      Beijing 2010-01-04 14:00:00     PM2.5  36.35  112.05  POWER_INDUSTRY       DEFAULT  0.000000e+00
7      Beijing 2010-01-04 14:00:00

In [ ]:
import pandas as pd

# Lese die CSV-Datei
df = pd.read_csv('hourly_profiles_china.csv')

# Filtere die Daten für activity_code "ENE" und "AGS" sowie month_id = 1
filtered_df = df[(df['activity_code'].isin(['ENE', 'AGS'])) & (df['month_id'] == 1)]

# Gib nur die Spalten h1 und activity_code aus
result = filtered_df
print(result)

    Country_code_A3 activity_code  month_id  Daytype_id        h1        h2  \
0               CHN           AGS         1           1  0.024845  0.024845   
1               CHN           AGS         1           2  0.033229  0.033229   
2               CHN           AGS         1           3  0.037441  0.037441   
216             CHN           ENE         1           1  0.032917  0.030000   
217             CHN           ENE         1           2  0.037292  0.035833   
218             CHN           ENE         1           3  0.039479  0.038750   

           h3        h4        h5        h6  ...       h15       h16  \
0    0.024845  0.024845  0.024845  0.026915  ...  0.070393  0.064182   
1    0.033229  0.033229  0.033229  0.034268  ...  0.056075  0.052960   
2    0.037441  0.037441  0.037441  0.037962  ...  0.048882  0.047322   
216  0.030000  0.029583  0.030833  0.033333  ...  0.047500  0.047083   
217  0.035833  0.035625  0.036250  0.037500  ...  0.044583  0.044375   
218  0.038750 

In [46]:
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd

dateipfad = 'edgar_extracted_monthly\edgar_all_pollutants_monthly.parquet'   # Pfad anpassen

# 1. Schema anzeigen
schema = pq.read_schema(dateipfad)
print("Alle Spaltennamen:", schema.names)

# 2. Dataset öffnen
dataset = ds.dataset(dateipfad, format='parquet')

# 3. Filter definieren
filter_year  = pc.field('year')  == 2010
filter_month = pc.field('month') == 1       # Achtung: 1, nicht '01'
filter_lat   = pc.field('lat')   == 36.35
filter_lon   = pc.field('lon')   == 113.05

# 4. Alle Bedingungen mit UND verknüpfen
filter_expr = filter_year & filter_month & filter_lat & filter_lon

# 5. Gefilterte Tabelle laden
table = dataset.to_table(filter=filter_expr)
df = table.to_pandas()

# 6. Ergebnis anzeigen
print(f"Gefundene Zeilen: {len(df)}")
print(df.to_string())   # zeigt alle Zeilen und alle Spalten
#Minibeispiel schreiben
table = pa.Table.from_pandas(df)
pq.write_table(table, 'MiniAllPolutantsMonthly.parquet', compression='snappy')

Alle Spaltennamen: ['city', 'year', 'month', 'lat', 'lon', 'emissions', 'pollutant', 'sector']
Gefundene Zeilen: 7
      city  year  month    lat     lon    emissions pollutant          sector
0  Beijing  2010      1  36.35  113.05    83.582748     PM2.5  POWER_INDUSTRY
1  Beijing  2010      1  36.35  113.05    53.058563     PM2.5       BUILDINGS
2  Beijing  2010      1  36.35  113.05     0.261683     PM2.5       TRANSPORT
3  Beijing  2010      1  36.35  113.05     3.908095        BC  POWER_INDUSTRY
4  Beijing  2010      1  36.35  113.05     5.574546        BC       BUILDINGS
5  Beijing  2010      1  36.35  113.05    25.148344        OC       BUILDINGS
6  Beijing  2010      1  36.35  113.05  1605.214233       SO2  POWER_INDUSTRY
